# 🧠 Omni-Modal Medical Diagnostic Framework

**GPU-accelerated training notebook for Google Colab**

This notebook provides:
1. ⚡ Environment setup (clone repo + install deps)
2. 🧪 Smoke test (validate pipeline with dummy data)
3. 🏋️ Full training (3-phase curriculum)
4. 📊 Evaluation and visualization

---
**Runtime**: Go to `Runtime → Change runtime type → GPU (T4)` before starting.

## 1️⃣ Setup Environment

In [ ]:
# Clone the repository
!git clone https://github.com/Ankit-blip737/Omni-Modal-Medical-Diagnostic-.git
%cd Omni-Modal-Medical-Diagnostic-

In [ ]:
# Install dependencies
!pip install -q torch torchvision torchaudio
!pip install -q transformers timm tqdm pyyaml scikit-learn matplotlib seaborn pandas tensorboard nibabel opencv-python

In [ ]:
# Verify GPU
import torch
print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Memory: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB")
else:
    print("⚠️ No GPU detected! Go to Runtime → Change runtime type → GPU")

## 2️⃣ Smoke Test (Dummy Data)

In [ ]:
import sys, os
sys.path.insert(0, '.')

from src.models.omni_modal import OmniModalFramework
from src.utils.logging import log_model_summary

# Build model (skip pretrained for speed)
model = OmniModalFramework(
    num_modalities=1,     # Single modality (X-ray)
    num_classes=14,       # CheXpert 14 labels
    text_model='pubmedbert',
    pretrained=True,
    freeze_image_backbone=True,
    freeze_text_layers=8,
    lora_rank=8,
)

log_model_summary(model)

device = 'cuda' if torch.cuda.is_available() else 'cpu'
model = model.to(device)
print(f"\n✅ Model loaded on {device}")

In [ ]:
# Quick forward pass test
import torch

B = 4  # batch size
dummy_images = torch.randn(B, 1, 1, 224, 224).to(device)  # 1 modality
dummy_ids = torch.randint(0, 1000, (B, 64)).to(device)
dummy_mask = torch.ones(B, 64, dtype=torch.long).to(device)

with torch.no_grad():
    output = model(
        images=dummy_images,
        input_ids=dummy_ids,
        attention_mask=dummy_mask,
        compute_alignment_loss=True,
    )

print("=== Forward Pass Output ===")
print(f"  Logits:          {output['logits'].shape}")         # (4, 14)
print(f"  Probabilities:   {output['probabilities'].shape}")  # (4, 14)
print(f"  Alignment loss:  {output['alignment_loss'].item():.4f}")
print(f"  Gate (visual):   {output['gate_values']['g_v']:.4f}")
print(f"  Gate (text):     {output['gate_values']['g_t']:.4f}")
print(f"\n✅ Forward pass successful!")

## 3️⃣ Training with Dummy Data

This section runs the full 3-phase training pipeline with synthetic data to verify everything works end-to-end on GPU.

In [ ]:
from src.data.mimic_cxr import get_mimic_dataloaders
from src.training.trainer import OmniModalTrainer
from src.training.losses import CombinedLoss

# Create dummy dataloaders (no real data needed)
data_config = {
    'root_dir': './data/mimic-cxr',  # Will auto-create dummy data
    'batch_size': 8,
    'num_workers': 2,
    'max_text_length': 128,  # Short for speed
    'pin_memory': True,
}
loaders = get_mimic_dataloaders(data_config)

print(f"Train batches: {len(loaders['train'])}")
print(f"Val batches:   {len(loaders['val'])}")

In [ ]:
# Configure training
training_config = {
    'mixed_precision': True,
    'gradient_clip_norm': 1.0,
    'log_every_n_steps': 5,
    # Phase configs (reduced for demo)
    'phase1': {
        'epochs': 3,
        'lr': 1e-4,
        'warmup_epochs': 1,
    },
    'phase2': {
        'epochs': 3,
        'lr': 5e-5,
        'weight_decay': 1e-4,
        'warmup_epochs': 1,
    },
    'phase3': {
        'epochs': 2,
        'lr': 1e-6,
        'weight_decay': 1e-5,
        'warmup_epochs': 1,
    },
}

# Create trainer
trainer = OmniModalTrainer(
    model=model,
    train_loader=loaders['train'],
    val_loader=loaders['val'],
    config=training_config,
    device=device,
    log_dir='./logs',
    checkpoint_dir='./checkpoints',
)

print("✅ Trainer created")

In [ ]:
# Phase 1: Contrastive Pre-training (alignment only)
trainer.train_phase1()

In [ ]:
# Phase 2: Fusion Training
trainer.train_phase2()

In [ ]:
# Phase 3: End-to-End Fine-tuning
trainer.train_phase3()

## 4️⃣ Evaluation

In [ ]:
import numpy as np
from src.evaluation.metrics import MetricTracker, format_metrics_table, compute_multilabel_metrics
from src.data.mimic_cxr import CHEXPERT_LABELS
from tqdm import tqdm

# Run evaluation
model.eval()
tracker = MetricTracker(class_names=CHEXPERT_LABELS)

with torch.no_grad():
    for batch in tqdm(loaders['val'], desc='Evaluating'):
        images = batch['images'].to(device)
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['labels']

        output = model(images, input_ids, attention_mask, compute_alignment_loss=False)
        probs = output['probabilities'].cpu().numpy()
        tracker.update(labels.numpy(), probs)

results = tracker.compute()
print(format_metrics_table(results))

## 5️⃣ Visualization

In [ ]:
from src.evaluation.visualization import plot_roc_curves, visualize_gate_values
import matplotlib.pyplot as plt

# ROC Curves
all_true = np.concatenate(tracker._all_true, axis=0)
all_prob = np.concatenate(tracker._all_prob, axis=0)

fig = plot_roc_curves(all_true, all_prob, CHEXPERT_LABELS)
plt.show()

In [ ]:
# Gate value analysis — shows modality weighting per sample
model.eval()
gate_vs, gate_ts = [], []

with torch.no_grad():
    for i, batch in enumerate(loaders['val']):
        if i >= 3:  # 3 batches
            break
        images = batch['images'].to(device)
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)

        output = model(images, input_ids, attention_mask)
        gate_vs.append(output['gate_values']['g_v'])
        gate_ts.append(output['gate_values']['g_t'])

gate_v = np.array(gate_vs)
gate_t = np.array(gate_ts)

fig = visualize_gate_values(gate_v, gate_t, sample_ids=[f'Batch {i}' for i in range(len(gate_v))])
plt.show()
print(f"\nAvg Visual Gate: {gate_v.mean():.4f}")
print(f"Avg Text Gate:   {gate_t.mean():.4f}")

## 6️⃣ Save & Download Model

In [ ]:
# Save final checkpoint
trainer.save_checkpoint('./checkpoints/final_model.pth', phase='all')
print("✅ Model saved to ./checkpoints/final_model.pth")

# Download from Colab (uncomment to use)
# from google.colab import files
# files.download('./checkpoints/final_model.pth')

## 7️⃣ Training with Real Data (MIMIC-CXR)

To train with real MIMIC-CXR data:

1. **Get access**: Apply at [PhysioNet](https://physionet.org/content/mimic-cxr/2.0.0/)
2. **Upload data** to Google Drive:
```
MyDrive/data/mimic-cxr/
├── train.csv
├── val.csv  
├── test.csv
└── images/
```
3. **Mount Drive** and update `root_dir`:

In [ ]:
# Uncomment to mount Google Drive for real data
# from google.colab import drive
# drive.mount('/content/drive')

# Then update data_config:
# data_config['root_dir'] = '/content/drive/MyDrive/data/mimic-cxr'
# loaders = get_mimic_dataloaders(data_config)

---
**📊 TensorBoard** (run in a separate cell):
```python
%load_ext tensorboard
%tensorboard --logdir ./logs
```